# Gold Layer — Insurance Star Schema
**Purpose:** Reshape clean Silver data into a star schema for fast, intuitive analytics.
**Output:** fact_charges + dim_region, dim_smoker, dim_bmi, dim_demographics
**Why:** Facts = numbers we measure (charges, children). Dimensions = labels we slice by.
Surrogate keys (small integers) link them — fast joins, narrow fact, stable IDs.

## Read Silver

In [1]:
# === READ INPUT ===
# Gold reads the SILVER table (clean + enriched). Each layer consumes the previous one.
silver = spark.read.table("silver_insurance")
print("Rows from silver:", silver.count())   # expect 1338

StatementMeta(, 1c2ca3b0-9d5e-46b7-b2b7-e8970b55a636, 3, Finished, Available, Finished, False)

Rows from silver: 1338


In [2]:
# === IMPORTS ===
# row_number -> generates sequential surrogate keys (1,2,3...)
# Window     -> defines the ordering row_number counts over
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

StatementMeta(, 1c2ca3b0-9d5e-46b7-b2b7-e8970b55a636, 4, Finished, Available, Finished, False)

## dim_region

In [3]:
# === DIMENSION: region ===
# Why .distinct(): a dimension holds each label ONCE (4 regions), not 1338 repeats.
# Why row_number().over(Window.orderBy(...)): assigns a fresh surrogate key 1,2,3,4.
dim_region = (
    silver.select("region").distinct()
          .withColumn("region_key", row_number().over(Window.orderBy("region")))
)
display(dim_region)

StatementMeta(, 1c2ca3b0-9d5e-46b7-b2b7-e8970b55a636, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 67fe4e0e-1ed0-4028-939a-1b6598d90cab)

## dim_smoker

In [4]:
# === DIMENSION: smoker ===
# Keyed on BOTH smoker (text) and is_smoker (flag) so the dimension carries both forms.
dim_smoker = (
    silver.select("smoker", "is_smoker").distinct()
          .withColumn("smoker_key", row_number().over(Window.orderBy("smoker")))
)
display(dim_smoker)

StatementMeta(, 1c2ca3b0-9d5e-46b7-b2b7-e8970b55a636, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0ebf5366-a00f-416d-9529-f53a424aa7db)

## dim_bmi

In [5]:
# === DIMENSION: bmi ===
# Keyed on bmi_category (the 4 bands). Raw bmi stays in the fact as a measure if needed.
dim_bmi = (
    silver.select("bmi_category").distinct()
          .withColumn("bmi_key", row_number().over(Window.orderBy("bmi_category")))
)
display(dim_bmi)

StatementMeta(, 1c2ca3b0-9d5e-46b7-b2b7-e8970b55a636, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9c31cc1b-d691-4efb-9258-1000135656cb)

## dim_demographics

In [6]:
# === DIMENSION: demographics ===
# Bundles the person's attributes: sex, age, age_band. Higher cardinality than the others
# (many age/sex combos) — that's fine; it's still far fewer than 1338 if ages repeat.
dim_demographics = (
    silver.select("sex", "age", "age_band").distinct()
          .withColumn("demographics_key", row_number().over(Window.orderBy("sex", "age")))
)
print("Demographics rows:", dim_demographics.count())
display(dim_demographics.limit(10))

StatementMeta(, 1c2ca3b0-9d5e-46b7-b2b7-e8970b55a636, 8, Finished, Available, Finished, False)

Demographics rows: 94


SynapseWidget(Synapse.DataFrame, 8fb55aed-d679-4d19-a3b2-d44f8fbfdfed)

## Build the fact

In [7]:
# === FACT: charges ===
# Move 1: join silver to each dimension on its NATURAL columns to pick up the surrogate key.
# Why LEFT join: keep every silver row even if (defensively) a match were missing.
# Move 2: SELECT only the keys + measures -> a narrow fact table, no repeated text labels.
fact_charges = (
    silver
    .join(dim_region,       on="region",                       how="left")
    .join(dim_smoker,       on=["smoker", "is_smoker"],         how="left")
    .join(dim_bmi,          on="bmi_category",                  how="left")
    .join(dim_demographics, on=["sex", "age", "age_band"],      how="left")
    .select(
        "region_key", "smoker_key", "bmi_key", "demographics_key",  # the links (foreign keys)
        "children",   # measure (a count)
        "charges"     # measure (the money — the star of the schema)
    )
)
print("Fact rows:", fact_charges.count())   # expect 1338 — one row per original record
display(fact_charges.limit(5))

StatementMeta(, 1c2ca3b0-9d5e-46b7-b2b7-e8970b55a636, 9, Finished, Available, Finished, False)

Fact rows: 1338


SynapseWidget(Synapse.DataFrame, 8084840f-bf42-4eb5-902a-253146899d51)

## Write all five tables

In [8]:
# === WRITE OUTPUT ===
# Naming convention: dim_* and fact_* signal "this is the Gold star schema" (Kimball style).
dim_region.write.mode("overwrite").format("delta").saveAsTable("dim_region")
dim_smoker.write.mode("overwrite").format("delta").saveAsTable("dim_smoker")
dim_bmi.write.mode("overwrite").format("delta").saveAsTable("dim_bmi")
dim_demographics.write.mode("overwrite").format("delta").saveAsTable("dim_demographics")
fact_charges.write.mode("overwrite").format("delta").saveAsTable("fact_charges")

print("Gold star schema saved: fact_charges + 4 dimensions.")

StatementMeta(, 1c2ca3b0-9d5e-46b7-b2b7-e8970b55a636, 10, Finished, Available, Finished, False)

Gold star schema saved: fact_charges + 4 dimensions.


## Verify by using the star

In [9]:
# === VERIFY: prove the star works with a real analytical join ===
# Question: average charge by region. We join the FACT to dim_region on the surrogate key,
# group by the label, and average the measure — exactly how Power BI will use this.
f = spark.read.table("fact_charges")
r = spark.read.table("dim_region")

(f.join(r, on="region_key", how="left")
   .groupBy("region")
   .avg("charges")
   .orderBy("region")
   .show())

StatementMeta(, 1c2ca3b0-9d5e-46b7-b2b7-e8970b55a636, 11, Finished, Available, Finished, False)

+---------+------------------+
|   region|      avg(charges)|
+---------+------------------+
|northeast|13406.384691358027|
|northwest|12417.575169230768|
|southeast| 14735.41153846154|
|southwest|12346.937907692307|
+---------+------------------+

